In [1]:
%load_ext autoreload
%autoreload 2

# Automatically discover candidate mappings for any network

This notebook creates provisional candidate mappings for `NETWORK_ID` from `meta_vars`. It uses variable names, standard names, descriptions, and units to find likely candidates for the six canonical variables.

Automatic discovery is a screening step, not final semantic validation. Review the candidates and their aggregation meaning before marking mappings as accepted.

## 1. Setup

In [2]:
import sys
function_path = '../func/'
sys.path.append(function_path)

In [3]:
from IPython.display import display
import sqlalchemy as sa
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path
from func_variable_mapping import (
    CANONICAL_VARIABLES, CANONICAL_RULES, save_candidate_mappings,
)


HERE = Path.cwd()
OUTPUT_DIR = HERE / 'outfile'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(HERE.parent / 'func'))

DB_URL = os.getenv(
    'CRMP_DB_URL',
    'postgresql://tongli1997@proddb01.pcic.uvic.ca,proddb02.pcic.uvic.ca/crmp'
    '?keepalives=1&keepalives_idle=300&keepalives_interval=300'
    '&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass',
)
engine = sa.create_engine(DB_URL, pool_pre_ping=True)
NETWORK_ID = 2
MIN_SCORE = 3

## 2. Load the selected network's variable catalog

In [4]:
catalog_query = sa.text("""
SELECT vars_id, network_id, net_var_name::text AS net_var_name, unit,
       standard_name, cell_method, display_name, short_name,
       long_description
FROM meta_vars
WHERE network_id = :network_id
ORDER BY standard_name
""")

network_catalog = pd.read_sql(
    catalog_query, engine, params={'network_id': NETWORK_ID}
)
display(network_catalog)

,vars_id,network_id,net_var_name,unit,standard_name,cell_method,display_name,short_name,long_description
0,725,2,stn_pres,hPa,air_pressure,time: point,Air Pressure,Air Pressure,NaN
1,439,2,ATMOSPHERIC_PRESSURE,millibar,air_pressure,time: point,Air Pressure (Point),air_pressure_point,NaN
2,733,2,mslp,hPa,air_pressure_at_mean_sea_level,time: point,Sea Level Pressure,slp,NaN
3,433,2,MAXIMUM_AIR_TEMPERATURE,celsius,air_temperature,time: maximum,Temperature (Max.),air_temperature_maximum,NaN
4,435,2,MINIMUM_AIR_TEMPERATURE,celsius,air_temperature,time: minimum,Temperature (Min.),air_temperature_minimum,NaN
5,434,2,CURRENT_AIR_TEMPERATURE1,celsius,air_temperature,time: point,Temperature (Point),air_temperature_point,NaN
6,436,2,CURRENT_AIR_TEMPERATURE2,celsius,air_temperature,time: point,Temperature (Point),air_temperature_point,NaN
7,562,2,Tn_Climatology,celsius,air_temperature,t: minimum within days t: mean within months t...,Temperature Climatology (Min.),air_temperaturet: minimum within days t: mean ...,Climatological mean of monthly mean minimum da...
8,561,2,Tx_Climatology,celsius,air_temperature,t: maximum within days t: mean within months t...,Temperature Climatology (Max.),air_temperaturet: maximum within days t: mean ...,Climatological mean of monthly mean maximum da...
9,563,2,T_mean_Climatology,celsius,air_temperature,t: mean within days t: mean within months t: m...,Temperature Climatology (Mean),air_temperaturet: mean within days t: mean wit...,Climatological mean of monthly mean of mean da...


## 3. Define transparent discovery rules

Positive patterns add evidence; exclusion patterns prevent common false matches. Adjust these expressions if the selected network uses unusual abbreviations.

In [5]:
DISCOVERY_RULES = {
    'air_temperature': {
        'include': [r'air.*temp', r'temp.*air', r'(^|_)temperature($|_)'],
        'exclude': [r'min', r'max', r'dew', r'soil', r'water', r'road', r'surface', r'climatology'],
        'daily_aggregation': 'mean',
    },
    'daily_min_temperature': {
        'include': [r'min(imum)?.*temp', r'temp.*min(imum)?'],
        'exclude': [r'soil', r'water', r'road', r'surface', r'climatology'],
        'daily_aggregation': 'min',
    },
    'daily_max_temperature': {
        'include': [r'max(imum)?.*temp', r'temp.*max(imum)?'],
        'exclude': [r'soil', r'water', r'road', r'surface', r'climatology'],
        'daily_aggregation': 'max',
    },
    'precipitation_amount': {
        'include': [r'precip', r'pcpn', r'(^|_)rain($|_)', r'rainfall'],
        'exclude': [r'snow', r'snw', r'climatology'],
        'daily_aggregation': 'sum',
    },
    'snowfall_amount': {
        'include': [r'snow.*fall', r'snwfl', r'standard.*snow'],
        'exclude': [r'depth', r'dpth', r'height', r'climatology'],
        'daily_aggregation': 'sum',
    },
    'snow_depth': {
        'include': [r'snow.*depth', r'depth.*snow', r'snw.*dpth', r'height.*snow'],
        'exclude': [r'fall', r'amount', r'climatology'],
        'daily_aggregation': 'fixed_hour',
    },
}

## 4. Score and rank candidates

In [6]:
def normalized_text(row, columns):
    return ' '.join(str(row.get(column) or '') for column in columns).lower()


def score_candidate(row, canonical_name):
    rule = DISCOVERY_RULES[canonical_name]
    name_text = normalized_text(
        row, ['net_var_name', 'short_name', 'display_name'])
    metadata_text = normalized_text(
        row, ['standard_name', 'long_description', 'cell_method'])
    all_text = name_text + ' ' + metadata_text

    if any(re.search(pattern, all_text) for pattern in rule['exclude']):
        return 0, 'excluded keyword'

    name_hits = sum(bool(re.search(pattern, name_text))
                    for pattern in rule['include'])
    metadata_hits = sum(bool(re.search(pattern, metadata_text))
                        for pattern in rule['include'])
    score = 3 * name_hits + 3 * metadata_hits

    expected_unit = CANONICAL_RULES[canonical_name].unit.lower()
    actual_unit = str(row.get('unit') or '').lower()
    unit_tokens = {
        'celsius': ['celsius', 'degc', 'degree_c', 'degrees_c', '°c'],
        'mm': ['mm', 'millimet'],
        'cm': ['cm', 'centimet'],
    }[expected_unit]
    unit_match = any(token in actual_unit for token in unit_tokens)
    if unit_match:
        score += 1

    evidence = f'name hits={name_hits}; metadata hits={metadata_hits}; unit match={unit_match}'
    return score, evidence


rows = []
for _, variable in network_catalog.iterrows():
    for canonical_name in CANONICAL_VARIABLES:
        score, evidence = score_candidate(variable, canonical_name)
        if score >= MIN_SCORE:
            rule = DISCOVERY_RULES[canonical_name]
            rows.append({
                'vars_id': int(variable['vars_id']),
                'net_var_name': variable['net_var_name'],
                'canonical_variable': canonical_name,
                'unit': variable['unit'],
                'daily_aggregation': rule['daily_aggregation'],
                'source_family': f'network{NETWORK_ID}_auto',
                'discovery_score': score,
                'discovery_evidence': evidence,
                'mapping_status': 'needs_review',
            })

discovered_candidates = pd.DataFrame(rows)
if not discovered_candidates.empty:
    discovered_candidates = discovered_candidates.sort_values(
        ['canonical_variable', 'discovery_score', 'vars_id'],
        ascending=[True, False, True],
    )
    discovered_candidates['priority'] = (
        discovered_candidates.groupby('canonical_variable').cumcount() + 1
    )
display(discovered_candidates)

,vars_id,net_var_name,canonical_variable,unit,daily_aggregation,source_family,discovery_score,discovery_evidence,mapping_status,priority
2,434,CURRENT_AIR_TEMPERATURE1,air_temperature,celsius,mean,network2_auto,13,name hits=3; metadata hits=1; unit match=True,needs_review,1
3,436,CURRENT_AIR_TEMPERATURE2,air_temperature,celsius,mean,network2_auto,13,name hits=3; metadata hits=1; unit match=True,needs_review,2
5,721,air_temp,air_temperature,celsius,mean,network2_auto,7,name hits=1; metadata hits=1; unit match=True,needs_review,3
0,433,MAXIMUM_AIR_TEMPERATURE,daily_max_temperature,celsius,max,network2_auto,10,name hits=2; metadata hits=1; unit match=True,needs_review,1
4,720,max_air_temp_snc_last_reset,daily_max_temperature,celsius,max,network2_auto,10,name hits=2; metadata hits=1; unit match=True,needs_review,2
1,435,MINIMUM_AIR_TEMPERATURE,daily_min_temperature,celsius,min,network2_auto,10,name hits=2; metadata hits=1; unit match=True,needs_review,1
6,722,min_air_temp_snc_last_reset,daily_min_temperature,celsius,min,network2_auto,10,name hits=2; metadata hits=1; unit match=True,needs_review,2
10,726,pcpn_amt_pst1hr,precipitation_amount,mm,sum,network2_auto,10,name hits=2; metadata hits=1; unit match=True,needs_review,1
9,732,pcpn_amt_pst24hrs,precipitation_amount,mm,sum,network2_auto,10,name hits=2; metadata hits=1; unit match=True,needs_review,2
11,440,PRECIPITATION_GAUGE_TOTAL,precipitation_amount,mm,sum,network2_auto,7,name hits=1; metadata hits=1; unit match=True,needs_review,3


## 5. Review and override

List false-positive IDs under `EXCLUDE`. Use `MANUAL_CANDIDATES` for variables missed by the patterns or to override inferred semantics. Re-run this cell after editing the lists.

In [7]:
# Exclusions are (vars_id, canonical_variable) pairs.
EXCLUDE = {
    (732, 'precipitation_amount')
}

# Add or replace reviewed candidates here.
MANUAL_CANDIDATES = [
    # {
    #     'vars_id': 123,
    #     'net_var_name': 'NETWORK_1_VARIABLE_NAME',
    #     'canonical_variable': 'air_temperature',
    #     'unit': 'celsius',
    #     'daily_aggregation': 'mean',
    #     'source_family': f'network{NETWORK_ID}_reviewed',
    #     'priority': 1,
    #     'mapping_status': 'candidate',
    # },
]

reviewed = discovered_candidates.copy()
if not reviewed.empty and EXCLUDE:
    excluded = pd.MultiIndex.from_tuples(EXCLUDE)
    row_keys = pd.MultiIndex.from_frame(
        reviewed[['vars_id', 'canonical_variable']])
    reviewed = reviewed.loc[~row_keys.isin(excluded)].copy()

if MANUAL_CANDIDATES:
    manual = pd.DataFrame(MANUAL_CANDIDATES)
    manual_keys = set(zip(manual['vars_id'], manual['canonical_variable']))
    if not reviewed.empty:
        keep = [
            (row.vars_id, row.canonical_variable) not in manual_keys
            for row in reviewed.itertuples()
        ]
        reviewed = reviewed.loc[keep]
    reviewed = pd.concat([reviewed, manual], ignore_index=True)

display(reviewed.sort_values(['canonical_variable', 'priority']))

,vars_id,net_var_name,canonical_variable,unit,daily_aggregation,source_family,discovery_score,discovery_evidence,mapping_status,priority
2,434,CURRENT_AIR_TEMPERATURE1,air_temperature,celsius,mean,network2_auto,13,name hits=3; metadata hits=1; unit match=True,needs_review,1
3,436,CURRENT_AIR_TEMPERATURE2,air_temperature,celsius,mean,network2_auto,13,name hits=3; metadata hits=1; unit match=True,needs_review,2
5,721,air_temp,air_temperature,celsius,mean,network2_auto,7,name hits=1; metadata hits=1; unit match=True,needs_review,3
0,433,MAXIMUM_AIR_TEMPERATURE,daily_max_temperature,celsius,max,network2_auto,10,name hits=2; metadata hits=1; unit match=True,needs_review,1
4,720,max_air_temp_snc_last_reset,daily_max_temperature,celsius,max,network2_auto,10,name hits=2; metadata hits=1; unit match=True,needs_review,2
1,435,MINIMUM_AIR_TEMPERATURE,daily_min_temperature,celsius,min,network2_auto,10,name hits=2; metadata hits=1; unit match=True,needs_review,1
6,722,min_air_temp_snc_last_reset,daily_min_temperature,celsius,min,network2_auto,10,name hits=2; metadata hits=1; unit match=True,needs_review,2
10,726,pcpn_amt_pst1hr,precipitation_amount,mm,sum,network2_auto,10,name hits=2; metadata hits=1; unit match=True,needs_review,1
11,440,PRECIPITATION_GAUGE_TOTAL,precipitation_amount,mm,sum,network2_auto,7,name hits=1; metadata hits=1; unit match=True,needs_review,3
8,441,PRECIPITATION_NEW,precipitation_amount,mm,sum,network2_auto,7,name hits=1; metadata hits=1; unit match=True,needs_review,4


## 6. Build and save this network's candidate mappings

The diagnostic columns remain in `discovered_candidates`; the final registry contains the same operational columns as the network-2 registry.

In [1]:
mapping_columns = [
    'vars_id', 'net_var_name', 'canonical_variable',
    'daily_aggregation', 'priority', 'source_family', 'mapping_status',
]
candidate_mappings = (
    reviewed[mapping_columns]
    .sort_values(['canonical_variable', 'priority', 'vars_id'])
    .reset_index(drop=True)
)
# display(candidate_mappings)

# Save or replace this network in candidate_mappings.csv.
all_saved_mappings = save_candidate_mappings(
    NETWORK_ID, candidate_mappings, OUTPUT_DIR / '0.candidate_mappings.csv'
)
display(all_saved_mappings)

NameError: name 'reviewed' is not defined

## 7. Use the registry in the canonical-variable notebook

In [9]:
# Example
CANONICAL_VARIABLE = 'precipitation_amount'
candidate_rows = candidate_mappings[
    candidate_mappings['canonical_variable'] == CANONICAL_VARIABLE
]
candidate_ids = candidate_rows['vars_id'].tolist()
display(candidate_rows)

,vars_id,net_var_name,canonical_variable,daily_aggregation,priority,source_family,mapping_status
7,726,pcpn_amt_pst1hr,precipitation_amount,sum,1,network2_auto,needs_review
8,440,PRECIPITATION_GAUGE_TOTAL,precipitation_amount,sum,3,network2_auto,needs_review
9,441,PRECIPITATION_NEW,precipitation_amount,sum,4,network2_auto,needs_review
10,442,HOURLY_PRECIPITATION,precipitation_amount,sum,5,network2_auto,needs_review
11,443,PRECIP_DETECTOR_RATIO,precipitation_amount,sum,6,network2_auto,needs_review
